# Quantization MobileNetV3 — Qualcomm AI Hub

Quantize student MobileNetV3 (đã train KD) sang **INT8** để deploy trên Snapdragon.

Flow theo official QAI Hub tutorial:

| Bước | Mô tả |
|------|-------|
| 1 | Setup dependencies (`qai-hub[torch]`) |
| 2 | Import & trace model (skip — dùng ONNX có sẵn từ KD export) |
| 3 | Select device & compile → TFLite (float32, để test trước khi quantize) |
| 4 | Submit inference job — test on-device với photometric sample |
| 5 | Profile on-device — đo latency & memory TRƯỚC khi quantize |
| 6 | Quantize: compile → ONNX → quantize INT8 → compile → TFLite |
| 7 | Validate accuracy quantized model (inference job + local onnxruntime) |
| 8 | Download optimized model |

**Input:** `kd_crossmodal_mobilenetv3_fr.onnx` (từ notebook KD export)  
**Cách dùng:** Sửa `CONFIGURATION` ở cell 3, chạy từ trên xuống.

## 1. Setup dependencies

In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')

REPO_URL    = 'https://github.com/NguyenXuanBinh22/DATN.git'
REPO_BRANCH = 'convnext-v2-dev'
REPO_DIR    = '/content/FR_Photometric_Stereo'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull origin {REPO_BRANCH}')
    print('Repo đã tồn tại, đã pull latest.')

%cd {REPO_DIR}
print(f'Working dir: {os.getcwd()}')

# qai-hub[torch] bao gồm PyTorch dependencies cần thiết cho hub
!pip install qai-hub
!pip install -q albumentations==1.3.1 timm tabulate onnxruntime
!pip install onnx

print('Setup xong.')

Mounted at /content/drive
/content/FR_Photometric_Stereo
Working dir: /content/FR_Photometric_Stereo
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.5/123.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.3/85.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.2/15.2 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.7/125.7 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 26.0 MB/s eta 0:00:00
Setup xong.


## 2. Imports & Cấu hình

In [ ]:
%cd /content/FR_Photometric_Stereo
import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import albumentations as A
import onnxruntime as ort
# import qai_hub as hub # Removed from here, moved to cell-config
from tabulate import tabulate

from going_modular.utils.roc_auc_id import compute_id_auc_gallery_probe, compute_rank1_gallery_probe
from going_modular.dataloader.multitask import create_eval_loaders

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
# ════════════════════════════════════════════════════════════
#  CẤU HÌNH — chỉnh sửa ở đây
# ════════════════════════════════════════════════════════════
from google.colab import userdata
import os
import qai_hub as hub # Added this import

# API token từ aihub.qualcomm.com → Account → API Token
QAI_HUB_TOKEN = userdata.get('QAI_HUB_TOKEN').strip() # Strip newline
os.environ['QAI_HUB_API_TOKEN'] = QAI_HUB_TOKEN # Set as environment variable

DRIVE_DATASET_DIR = '/content/drive/MyDrive/Photometric_DB_Full/'

# File .onnx đã export từ notebook KD (.onnx và .onnx.data phải cùng thư mục)
ONNX_PATH = (
    '/content/drive/MyDrive/experiments/'
    '(1-15-100-200)new_lan2_KD_CrossModal_ConcatTeacher_to_MobileNetV3_Albedo/'
    'checkpoints/kd_crossmodal_mobilenetv3_fr.onnx'
)

OUTPUT_DIR       = '/content/drive/MyDrive/experiments/quantized_mobilenetv3/'
STUDENT_MODALITY = 'albedo'   # 'albedo' hoặc 'normalmap'
TARGET_DEVICE    = 'Samsung Galaxy S24 (Family)'
INPUT_SHAPE      = (1, 3, 112, 112)

# Số ảnh calibration (khuyến nghị 200-500)
NUM_CALIBRATION_SAMPLES = 200

CONFIGURATION = {
    'dataset_dir': DRIVE_DATASET_DIR,
    'type':        STUDENT_MODALITY,
    'image_size':  112,
    'batch_size':  16,
    'num_workers': 2,
    'backbone':    'mobilenetv3_large_100',
    'num_classes': None,
    'use_sampler': False,
    'device':      device,
    'output_dir':  OUTPUT_DIR,
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
!qai-hub configure --api_token {QAI_HUB_TOKEN} 
qai_device = hub.Device(TARGET_DEVICE)


print(f'ONNX path     : {ONNX_PATH}')
print(f'Output dir    : {OUTPUT_DIR}')
print(f'Modality      : {STUDENT_MODALITY}')
print(f'Target device : {TARGET_DEVICE}')
print(f'ONNX exists   : {os.path.exists(ONNX_PATH)}')

2026-06-15 02:10:15.201 - INFO - Enabling verbose logging.
qai-hub configuration saved to /root/.qai_hub/client.ini
==================== /root/.qai_hub/client.ini ====================
[api]
api_token = fumjf9svarjyokl2jlpcnja3561tu1zu604c1fus
api_url = https://workbench.aihub.qualcomm.com
web_url = https://workbench.aihub.qualcomm.com
verbose = True
client_mode = cli


ONNX path     : /content/drive/MyDrive/experiments/(1-15-100-200)new_lan2_KD_CrossModal_ConcatTeacher_to_MobileNetV3_Albedo/checkpoints/kd_crossmodal_mobilenetv3_fr.onnx
Output dir    : /content/drive/MyDrive/experiments/quantized_mobilenetv3/
Modality      : albedo
Target device : Samsung Galaxy S24 (Family)
ONNX exists   : True


In [4]:
import onnx

print('Merging external weights into single ONNX file...')
model_proto = onnx.load(ONNX_PATH, load_external_data=True)

ONNX_PATH_MERGED = ONNX_PATH.replace('.onnx', '_merged.onnx')
onnx.save_model(model_proto, ONNX_PATH_MERGED, save_as_external_data=False)

print(f'Done: {ONNX_PATH_MERGED}')
print(f'Size: {os.path.getsize(ONNX_PATH_MERGED) / 1024 / 1024:.1f} MB')

Merging external weights into single ONNX file...
Done: /content/drive/MyDrive/experiments/(1-15-100-200)new_lan2_KD_CrossModal_ConcatTeacher_to_MobileNetV3_Albedo/checkpoints/kd_crossmodal_mobilenetv3_fr_merged.onnx
Size: 13.6 MB


## 3. Import & Trace Model

> **Skip bước này** — model đã được trace và export ONNX trong notebook KD (`cell-export`).  
> Với custom model, QAI Hub khuyến nghị dùng ONNX trực tiếp thay vì TorchScript.
>
> Nếu muốn trace lại từ checkpoint `.pth`:

In [5]:
# # (Tùy chọn) Re-export ONNX từ checkpoint nếu cần
# # Thường không cần — dùng file .onnx đã có từ notebook KD

# from going_modular.model.FaceRecognitionMobileNetV3 import FaceRecognitionMobileNetV3

# STUDENT_CKPT = '/content/drive/MyDrive/experiments/.../checkpoints/best_model.pth'
# df_train = pd.read_csv(os.path.join(DRIVE_DATASET_DIR, 'train_split.csv'))
# num_classes = int(df_train['id'].max() + 1)

# student = FaceRecognitionMobileNetV3(num_classes=num_classes, backbone='mobilenetv3_large_100')
# ckpt = torch.load(STUDENT_CKPT, map_location='cpu', weights_only=False)
# student.load_state_dict(ckpt['model_state_dict'])

# class InferenceWrapper(nn.Module):
#     def __init__(self, model):
#         super().__init__()
#         self.backbone  = model.backbone
#         self.embedding = model.embedding
#     def forward(self, x):
#         return F.normalize(self.embedding(self.backbone(x)), p=2, dim=1)

# inference_model = InferenceWrapper(student).eval().cpu()
# dummy_input = torch.randn(*INPUT_SHAPE)
# torch.onnx.export(
#     inference_model, dummy_input, ONNX_PATH,
#     input_names=['input'], output_names=['embedding'],
#     dynamic_axes={'input': {0: 'batch'}, 'embedding': {0: 'batch'}},
#     opset_version=17,
# )
# print(f'Re-exported ONNX: {ONNX_PATH}')

# print('Dùng ONNX đã có sẵn:', ONNX_PATH)

## 4. Select Device & Compile (Float32)

Compile ONNX → TFLite (float32) trước để test baseline trên device,  
sau đó mới quantize nếu cần giảm memory / latency.

In [6]:
print('Compiling ONNX → TFLite (float32)...')

compile_float_job = hub.submit_compile_job(
    model=ONNX_PATH_MERGED,
    device=qai_device,
    input_specs=dict(input=INPUT_SHAPE),
    options='--target_runtime tflite',
)
assert isinstance(compile_float_job, hub.CompileJob)

float_tflite_model = compile_float_job.get_target_model()
assert isinstance(float_tflite_model, hub.Model)
print(f'Float32 TFLite model ID: {float_tflite_model.model_id}')

Compiling ONNX → TFLite (float32)...
Uploading kd_crossmodal_mobilenetv3_fr_merged.onnx


100%|██████████| 13.6M/13.6M [00:00<00:00, 15.8MB/s]


Scheduled compile job (j5qwjx2m5) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/j5qwjx2m5/

Waiting for compile job (j5qwjx2m5) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          
Float32 TFLite model ID: mnodwvj7m


## 5. Submit Inference Job — Test On-Device

Test model trên thiết bị thật với 1 ảnh photometric sample.  
Kiểm tra output shape và giá trị trước khi đánh giá toàn bộ.

In [7]:
dataset_dir = CONFIGURATION['dataset_dir']
probe_csv   = os.path.join(dataset_dir, 'probe_split.csv')
df_probe    = pd.read_csv(probe_csv)

file_map = {
    'albedo':    'albedo_map_new_crop.exr.npy',
    'normalmap': 'normal_map_new_crop.exr.npy',
}
file_suffix   = file_map[STUDENT_MODALITY]
infer_transform = A.Compose([A.Resize(112, 112)])

# Load 1 ảnh sample để test inference job
sample_row  = df_probe.iloc[0]
sample_path = os.path.join(dataset_dir, str(sample_row['id']),
                           str(sample_row['session']), file_suffix)

img = np.load(sample_path)
if img.ndim == 3 and img.shape[0] == 3:
    img = img.transpose(1, 2, 0)                 # CHW → HWC
img = infer_transform(image=img.astype(np.float32))['image']
img = np.expand_dims(np.transpose(img, (2, 0, 1)), 0)  # [1, 3, 112, 112]

print(f'Sample input shape: {img.shape} | range: [{img.min():.3f}, {img.max():.3f}]')

print('Submitting inference job (float32 TFLite)...')
inference_job = hub.submit_inference_job(
    model=float_tflite_model,
    device=qai_device,
    inputs=dict(input=[img]),
)
assert isinstance(inference_job, hub.InferenceJob)

on_device_output = inference_job.download_output_data()
output_name      = list(on_device_output.keys())[0]
embedding_ondevice = on_device_output[output_name][0]   # [1, 512]

print(f'On-device output shape : {embedding_ondevice.shape}')
print(f'Embedding norm         : {np.linalg.norm(embedding_ondevice):.4f}')  # ≈ 1.0 (L2 norm)

Sample input shape: (1, 3, 112, 112) | range: [0.076, 0.629]
Submitting inference job (float32 TFLite)...


Uploading dataset: 142kB [00:00, 831kB/s]                    


Scheduled inference job (j56vk917p) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/j56vk917p/

Waiting for inference job (j56vk917p) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          


tmp151e4c6r.h5: 100%|██████████| 13.9k/13.9k [00:00<00:00, 16.1MB/s]

On-device output shape : (1, 512)
Embedding norm         : 1.0000


## 6. Profile On-Device (Float32)

Đo latency & memory của float32 model trước khi quyết định quantize.

In [8]:
print('Submitting profile job (float32 TFLite)...')

profile_float_job = hub.submit_profile_job(
    model=float_tflite_model,
    device=qai_device,
)
assert isinstance(profile_float_job, hub.ProfileJob)

profile_float = profile_float_job.download_profile()
summary_float = profile_float['execution_summary']

rows = [
    ['Model',            'MobileNetV3 TFLite (float32)'],
    ['Device',           TARGET_DEVICE],
    ['Inference time',   f"{summary_float.get('estimated_inference_time', '?')} ms"],
    ['Peak memory',      f"{summary_float.get('peak_memory_bytes', '?')} bytes"],
]
print(tabulate(rows, tablefmt='fancy_grid'))

Submitting profile job (float32 TFLite)...
Scheduled profile job (jgzw1q76g) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jgzw1q76g/

Waiting for profile job (jgzw1q76g) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          
╒════════════════╤══════════════════════════════╕
│ Model          │ MobileNetV3 TFLite (float32) │
├────────────────┼──────────────────────────────┤
│ Device         │ Samsung Galaxy S24 (Family)  │
├────────────────┼──────────────────────────────┤
│ Inference time │ 206 ms                       │
├────────────────┼──────────────────────────────┤
│ Peak memory    │ ? bytes                      │
╘════════════════╧══════════════════════════════╛


## 7. Quantize Model (INT8)

### 7.1 Compile ONNX → Optimized ONNX

Bắt buộc dù ONNX đã có: compiler chạy optimization pass trước quantize.

In [10]:
print('Compiling ONNX → optimized ONNX...')

compile_onnx_job = hub.submit_compile_job(
    model=ONNX_PATH_MERGED,
    device=qai_device,
    input_specs=dict(input=INPUT_SHAPE),
    options='--target_runtime onnx',
)
assert isinstance(compile_onnx_job, hub.CompileJob)

unquantized_onnx_model = compile_onnx_job.get_target_model()
assert isinstance(unquantized_onnx_model, hub.Model)
print(f'Optimized ONNX model ID: {unquantized_onnx_model.model_id}')

Compiling ONNX → optimized ONNX...
Uploading kd_crossmodal_mobilenetv3_fr_merged.onnx


100%|██████████| 13.6M/13.6M [00:00<00:00, 16.1MB/s]


Scheduled compile job (jp13rkd25) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jp13rkd25/

Waiting for compile job (jp13rkd25) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          
Optimized ONNX model ID: mn124klpm


### 7.2 Load & Pre-process Calibration Data

Dùng **probe set** (~200 samples) — representative với inference thực tế.  
Load từ `.npy` photometric data (không phải ảnh RGB thông thường).

In [11]:
train_csv = os.path.join(dataset_dir, 'train_split.csv')
df_train  = pd.read_csv(train_csv)
CONFIGURATION['num_classes'] = int(df_train['id'].max() + 1)

df_calib = df_probe.sample(n=min(NUM_CALIBRATION_SAMPLES, len(df_probe)), random_state=42)
print(f'Calibration samples: {len(df_calib)} / {len(df_probe)} (probe set)')

sample_inputs = []
skipped = 0

for _, row in df_calib.iterrows():
    npy_path = os.path.join(dataset_dir, str(row['id']), str(row['session']), file_suffix)
    try:
        img = np.load(npy_path)
        if img.ndim == 3 and img.shape[0] == 3:
            img = img.transpose(1, 2, 0)                      # CHW → HWC
        img = infer_transform(image=img.astype(np.float32))['image']
        img = np.expand_dims(np.transpose(img, (2, 0, 1)), 0) # [1, 3, 112, 112]
        sample_inputs.append(img)
    except Exception:
        skipped += 1

print(f'Loaded: {len(sample_inputs)} | skipped: {skipped}')
print(f'Value range: [{sample_inputs[0].min():.3f}, {sample_inputs[0].max():.3f}]')

# Key phải khớp với input_name trong ONNX ('input')
calibration_data = dict(input=sample_inputs)

Calibration samples: 200 / 288 (probe set)
Loaded: 200 | skipped: 0
Value range: [0.089, 0.860]


### 7.3 Submit Quantize Job

In [12]:
print('Submitting quantize job (INT8 w8a8)...')

quantize_job = hub.submit_quantize_job(
    model=unquantized_onnx_model,
    calibration_data=calibration_data,
    weights_dtype=hub.QuantizeDtype.INT8,
    activations_dtype=hub.QuantizeDtype.INT8,
)
assert isinstance(quantize_job, hub.QuantizeJob)

quantized_onnx_model = quantize_job.get_target_model()
assert isinstance(quantized_onnx_model, hub.Model)
print(f'Quantized ONNX model ID: {quantized_onnx_model.model_id}')

Submitting quantize job (INT8 w8a8)...


Uploading dataset: 26.0MB [00:01, 23.1MB/s]                            


Scheduled quantize job (jpr9zm49p) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jpr9zm49p/

Waiting for quantize job (jpr9zm49p) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          
Quantized ONNX model ID: mqv73zojq


### 7.4 Compile Quantized ONNX → Target Runtime

In [13]:
# ── TFLite (default) ──────────────────────────────────────────
print('Compiling quantized ONNX → TFLite (INT8)...')
compile_quant_job = hub.submit_compile_job(
    model=quantized_onnx_model,
    device=qai_device,
    options='--target_runtime tflite --quantize_io',
)
assert isinstance(compile_quant_job, hub.CompileJob)
quant_tflite_model = compile_quant_job.get_target_model()
print(f'Quantized TFLite model ID: {quant_tflite_model.model_id}')

# ── QNN Context Binary (uncomment để dùng thay thế) ──────────
# compile_qnn_job = hub.submit_compile_job(
#     model=quantized_onnx_model,
#     device=qai_device,
#     options='--target_runtime qnn_context_binary --quantize_io',
# )
# quant_qnn_model = compile_qnn_job.get_target_model()
# print(f'Quantized QNN model ID: {quant_qnn_model.model_id}')

Compiling quantized ONNX → TFLite (INT8)...
Scheduled compile job (jpyn9k47g) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jpyn9k47g/

Waiting for compile job (jpyn9k47g) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          
Quantized TFLite model ID: mn07w62xm


## 8. Validate Accuracy & Performance (Quantized)

Lặp lại các bước 5–6 cho quantized model.

In [ ]:
# ── Inference job: so sánh output float32 vs INT8 ─────────────
print('Submitting inference job (quantized TFLite INT8)...')
inference_quant_job = hub.submit_inference_job(
    model=quant_tflite_model,
    device=qai_device,
    inputs=dict(input=[img]),   # cùng ảnh sample đã dùng ở bước 5
)
assert isinstance(inference_quant_job, hub.InferenceJob)

on_device_quant_output = inference_quant_job.download_output_data()
output_name_q          = list(on_device_quant_output.keys())[0]
embedding_quant        = on_device_quant_output[output_name_q][0]   # [1, 512]

# Cosine similarity giữa 2 embedding (float32 vs INT8)
emb_f32  = embedding_ondevice.flatten()
emb_int8 = embedding_quant.flatten()
cos_sim  = np.dot(emb_f32, emb_int8) / (np.linalg.norm(emb_f32) * np.linalg.norm(emb_int8) + 1e-8)

rows = [
    ['Embedding shape (float32)', str(embedding_ondevice.shape)],
    ['Embedding shape (INT8)',    str(embedding_quant.shape)],
    ['Cosine sim (f32 vs INT8)',  f'{cos_sim:.4f}'],  # ≈ 1.0 nếu quantize không làm hỏng
]
print(tabulate(rows, tablefmt='fancy_grid'))

In [ ]:
# ── Profile job: so sánh latency float32 vs INT8 ─────────────
print('Submitting profile job (quantized TFLite INT8)...')
profile_quant_job = hub.submit_profile_job(
    model=quant_tflite_model,
    device=qai_device,
)
assert isinstance(profile_quant_job, hub.ProfileJob)

profile_quant   = profile_quant_job.download_profile()
summary_quant   = profile_quant['execution_summary']

compare_rows = [
    ['Model',          'Float32 TFLite',                                         'INT8 TFLite'],
    ['Inference time', f"{summary_float.get('estimated_inference_time', '?')} ms",
                       f"{summary_quant.get('estimated_inference_time', '?')} ms"],
    ['Peak memory',    f"{summary_float.get('peak_memory_bytes', '?')} bytes",
                       f"{summary_quant.get('peak_memory_bytes', '?')} bytes"],
]
print(f'\n--- Performance: Float32 vs INT8 ({TARGET_DEVICE}) ---')
print(tabulate(compare_rows, headers=['Metric', 'Float32', 'INT8'], tablefmt='fancy_grid'))

In [ ]:
# ── AUC accuracy: float32 vs INT8 dùng onnxruntime local ─────
import zipfile

class OnnxModelWrapper(nn.Module):
    """Wrap onnxruntime session — interface get_embedding() cho compute_id_auc."""
    def __init__(self, onnx_path: str):
        super().__init__()
        self.session    = ort.InferenceSession(
            onnx_path,
            providers=['CUDAExecutionProvider', 'CPUExecutionProvider'],
        )
        self.input_name = self.session.get_inputs()[0].name

    def get_embedding(self, x: torch.Tensor) -> torch.Tensor:
        out = self.session.run(None, {self.input_name: x.cpu().numpy().astype(np.float32)})[0]
        return torch.from_numpy(out)

    def forward(self, x):
        return self.get_embedding(x)


# Download quantized ONNX về local
# QAI Hub download() tự append model type + '.zip' vào path được truyền vào.
# Truyền path KHÔNG có '.onnx' để file thực sự là <base>.onnx.zip
quant_onnx_base  = os.path.join(OUTPUT_DIR, 'kd_crossmodal_mobilenetv3_int8')
quant_onnx_zip   = quant_onnx_base + '.onnx.zip'
quant_onnx_local = quant_onnx_base + '.onnx'

if not os.path.exists(quant_onnx_local):
    quantized_onnx_model.download(quant_onnx_base)   # → saves as <base>.onnx.zip

    # Zip chứa 2 file: model.onnx (cấu trúc) + model.data (weights)
    # Phải extract CẢ HAI vì ONNX reference model.data theo đường dẫn tương đối
    with zipfile.ZipFile(quant_onnx_zip, 'r') as z:
        z.extractall(OUTPUT_DIR)

    # Đổi tên model.onnx → tên mong muốn; model.data giữ nguyên (ONNX trỏ tới nó)
    extracted_onnx = os.path.join(OUTPUT_DIR, 'model.onnx')
    if os.path.exists(extracted_onnx):
        os.rename(extracted_onnx, quant_onnx_local)

    print(f'Downloaded & extracted quantized ONNX: {quant_onnx_local}')
    print(f'model.data exists: {os.path.exists(os.path.join(OUTPUT_DIR, "model.data"))}')
else:
    print(f'Dùng cached quantized ONNX: {quant_onnx_local}')

onnx_size  = os.path.getsize(quant_onnx_local) / 1024 / 1024
data_path  = os.path.join(OUTPUT_DIR, 'model.data')
data_size  = os.path.getsize(data_path) / 1024 / 1024 if os.path.exists(data_path) else 0
print(f'Size: model.onnx={onnx_size:.1f} MB, model.data={data_size:.1f} MB')

# Eval loaders
eval_transform = A.Compose([A.Resize(112, 112)])
train_csv = os.path.join(dataset_dir, 'train_split.csv')
df_train  = pd.read_csv(train_csv)
CONFIGURATION['num_classes'] = int(df_train['id'].max() + 1)
gallery_dl, probe_dl = create_eval_loaders(CONFIGURATION, eval_transform)

# Float32 ONNX (dùng merged để tránh phụ thuộc external data file)
print('Evaluating float32 ONNX...')
orig_model = OnnxModelWrapper(ONNX_PATH_MERGED)
orig_auc   = compute_id_auc_gallery_probe(gallery_dl, probe_dl, orig_model, device)
orig_rank1 = compute_rank1_gallery_probe(gallery_dl, probe_dl, orig_model, device)

# INT8 ONNX
print('Evaluating INT8 quantized ONNX...')
quant_model = OnnxModelWrapper(quant_onnx_local)
quant_auc   = compute_id_auc_gallery_probe(gallery_dl, probe_dl, quant_model, device)
quant_rank1 = compute_rank1_gallery_probe(gallery_dl, probe_dl, quant_model, device)

acc_rows = [
    ['Cosine AUC (gallery→probe)',
     f"{orig_auc['id_cosine']:.4f}",
     f"{quant_auc['id_cosine']:.4f}",
     f"{orig_auc['id_cosine'] - quant_auc['id_cosine']:+.4f}"],
    ['Euclidean AUC (gallery→probe)',
     f"{orig_auc['id_euclidean']:.4f}",
     f"{quant_auc['id_euclidean']:.4f}",
     f"{orig_auc['id_euclidean'] - quant_auc['id_euclidean']:+.4f}"],
    ['Rank-1 Acc (gallery→probe)',
     f'{orig_rank1:.4f}',
     f'{quant_rank1:.4f}',
     f'{orig_rank1 - quant_rank1:+.4f}'],
]
print('\n--- Accuracy: Float32 vs INT8 ---')
print(tabulate(acc_rows,
               headers=['Metric', 'Float32', 'INT8', 'Drop'],
               tablefmt='fancy_grid'))

## 9. Download Optimized Model

Download model đã quantize & compile về Google Drive.

In [ ]:
import zipfile

tflite_base = os.path.join(OUTPUT_DIR, 'kd_crossmodal_mobilenetv3_int8')
tflite_path = tflite_base + '.tflite'

if not os.path.exists(tflite_path):
    quant_tflite_model.download(tflite_base)   # hub appends .tflite (or .tflite.zip)
    # Nếu hub download dưới dạng zip, extract ra
    zip_path = tflite_base + '.tflite.zip'
    if os.path.exists(zip_path) and not os.path.exists(tflite_path):
        with zipfile.ZipFile(zip_path, 'r') as z:
            tflite_name = next(n for n in z.namelist() if n.endswith('.tflite'))
            data = z.read(tflite_name)
        with open(tflite_path, 'wb') as f:
            f.write(data)

print(f'Downloaded: {tflite_path}')
print(f'File size : {os.path.getsize(tflite_path) / 1024:.1f} KB')

## 10. Lưu Model IDs (tránh resubmit khi disconnect)

In [ ]:
model_ids = {
    'float32_tflite_model_id':   float_tflite_model.model_id,
    'unquantized_onnx_model_id': unquantized_onnx_model.model_id,
    'quantized_onnx_model_id':   quantized_onnx_model.model_id,
    'quant_tflite_model_id':     quant_tflite_model.model_id,
    'target_device':             TARGET_DEVICE,
    'student_modality':          STUDENT_MODALITY,
    'source_onnx':               ONNX_PATH,
}

ids_path = os.path.join(OUTPUT_DIR, 'qai_hub_model_ids.json')
with open(ids_path, 'w') as f:
    json.dump(model_ids, f, indent=2)

print(f'Saved: {ids_path}')
print(json.dumps(model_ids, indent=2))

## 11. Reload từ ID đã lưu (nếu session bị ngắt)

In [ ]:
# ids_path = os.path.join(OUTPUT_DIR, 'qai_hub_model_ids.json')
# with open(ids_path) as f:
#     saved_ids = json.load(f)

# float_tflite_model     = hub.get_model(saved_ids['float32_tflite_model_id'])
# unquantized_onnx_model = hub.get_model(saved_ids['unquantized_onnx_model_id'])
# quantized_onnx_model   = hub.get_model(saved_ids['quantized_onnx_model_id'])
# quant_tflite_model     = hub.get_model(saved_ids['quant_tflite_model_id'])
# print('Models reloaded.')